# Deteksi Deepfake Audio pada Fake-or-Real, di Google Colab

Notebook ini menjalankan repositori tesis `general-ai` di Colab. Ada dua jalur
dan keduanya berdiri sendiri.

| | **Jalur A, analisis** | **Jalur B, pelatihan** |
|---|---|---|
| Butuh GPU | tidak | ya |
| Butuh dataset | **tidak** | ya, 1 GB diunduh |
| Lama | sekitar 5 menit | 20 menit sampai beberapa jam |
| Isi | seluruh tabel hasil, sembilan gambar, `NASKAH.pdf` | melatih model dari nol lalu membandingkannya |

Jalur A bisa berjalan tanpa dataset karena **skor uji 159 run sudah ikut
ter-commit** di repositori (`runs/*/test_scores.npy`, seluruhnya hanya 17 MB).
Bobot modelnya yang berukuran 120 GB memang tidak ikut, tetapi seluruh
analisis, pengujian statistik, gambar, dan naskah dihitung dari skor itu,
bukan dari bobotnya.

**Jalankan sel secara berurutan.** Bagian 0 sampai 3 wajib untuk kedua jalur.


---
## 0. Periksa runtime

Jalankan ini lebih dulu. Untuk Jalur B, atur runtime ke GPU melalui
**Runtime → Change runtime type → T4 GPU** sebelum menjalankan sel ini.


In [ ]:
import os, shutil, sys

print("Python :", sys.version.split()[0])
try:
    import torch
    print("PyTorch:", torch.__version__)
    if torch.cuda.is_available():
        cc = torch.cuda.get_device_capability(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU    : {torch.cuda.get_device_name(0)}  "
              f"(compute capability {cc[0]}.{cc[1]}, {vram:.1f} GB)")
        if cc[0] >= 8:
            print("         bfloat16 didukung perangkat keras, tanpa patch pun aman")
        else:
            print("         bfloat16 TIDAK ada di perangkat keras kartu ini.")
            print("         Bagian 8 memasang patch supaya memakai float16, wajib "
                  "untuk Jalur B.")
    else:
        print("GPU    : tidak ada. Jalur A tetap bisa jalan, Jalur B tidak.")
except ImportError:
    print("PyTorch belum terpasang, akan dipasang di bagian 2")

t, _, f = shutil.disk_usage("/content")
print(f"Disk   : {f/1e9:.0f} GB bebas dari {t/1e9:.0f} GB")
print("CPU    :", os.cpu_count(), "core")


---
## 1. Masukkan kodenya

Repositori <https://github.com/Tristan-tech-ai/general-AI> bersifat publik,
jadi **Cara 1 cukup untuk hampir semua keadaan**. Dua cara lain disediakan
sebagai cadangan dan tidak perlu dijalankan.

### Cara 1, klon dari GitHub

Tidak perlu token, tidak perlu mengunggah apa pun, dan berpindah device tinggal
menjalankan sel ini lagi. Klonnya sekitar 25 MB dan memakan belasan detik.

Skor uji 159 run ikut di dalam repositori, jadi seluruh Jalur A dapat berjalan
langsung setelah sel ini selesai. Bobot model yang berukuran 120 GB tidak ikut
dan memang tidak diperlukan.


In [ ]:
import glob, os, subprocess

AKAR = "/content/general-ai"
os.chdir("/content")

if os.path.exists(AKAR):
    print(AKAR, "sudah ada, klon dilewati")
else:
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/Tristan-tech-ai/general-AI.git",
                    AKAR], check=True)
    print("berhasil diklon ke", AKAR)

os.chdir(AKAR)
subprocess.run(["git", "log", "--oneline", "-1"])
print("\ncwd    :", os.getcwd())
print("skrip  :", len(glob.glob("*.py")), "berkas python")
print("run    :", len(glob.glob("runs/*")), "di runs/,",
      len(glob.glob("runs_augbug/*")), "di runs_augbug/,",
      len(glob.glob("runs_pra_perbaikan/*")), "di runs_pra_perbaikan/")
print("skor   :", len(glob.glob("runs*/*/test_scores.npy")), "berkas test_scores.npy")
print("hasil  :", len(glob.glob("*.md")), "dokumen markdown")
print("bobot  :", len(glob.glob("runs*/*/best.pt")), "berkas best.pt "
      "(memang nol, 120 GB itu tidak ikut dan tidak diperlukan)")


### Cara 2, unggah berkas zip

**Lewati sel ini kalau Cara 1 sudah berhasil.** Berguna hanya bila internet
Colab tidak dapat menjangkau GitHub, atau bila Anda ingin memakai keadaan
lokal yang belum sempat di-commit.

Bundle-nya dibuat ulang dengan `py colab/buat_bundle.py` di komputer Anda, dan
selalu diambil dari commit terakhir, bukan dari working tree.


In [ ]:
# Sel cadangan. Lewati bila Cara 1 sudah berhasil.
import glob, os, zipfile

AKAR = "/content/general-ai"
os.chdir("/content")
if not os.path.exists(AKAR):
    from google.colab import files
    print("Pilih colab_bundle.zip dari komputer Anda ...")
    naik = files.upload()
    nama = list(naik)[0]
    with zipfile.ZipFile(nama) as z:
        z.extractall(AKAR)
    os.remove(nama)
os.chdir(AKAR)
print("cwd:", os.getcwd(), "|", len(glob.glob("runs*/*/test_scores.npy")), "skor run")


### Cara 3, dari Google Drive

**Lewati sel ini juga.** Berguna kalau Anda sudah menyalin folder proyek ke
Drive dan ingin memakai versi itu.


In [ ]:
# Sel cadangan. Lewati bila Cara 1 sudah berhasil.
import os, shutil
from google.colab import drive

drive.mount("/content/drive")
SUMBER = "/content/drive/MyDrive/general-ai"     # sesuaikan
shutil.copytree(SUMBER, "/content/general-ai", dirs_exist_ok=True)
os.chdir("/content/general-ai")
print("cwd:", os.getcwd())


---
## 2. Pasang dependensi

Colab sudah membawa torch, torchaudio, numpy, scipy, scikit-learn, matplotlib,
dan Pillow. Yang perlu ditambahkan hanya `reportlab` untuk membangun PDF dan
`soundfile` untuk membaca audio.

Repositori ini sengaja **tidak** memakai `librosa` maupun `numba`, jadi tidak
ada dependensi yang rewel di sini.


In [ ]:
!pip install --quiet reportlab soundfile
!pip install --quiet --upgrade transformers

import importlib
print()
for m in ["torch", "torchaudio", "transformers", "soundfile", "numpy", "scipy",
          "sklearn", "matplotlib", "reportlab", "PIL"]:
    try:
        mod = importlib.import_module(m)
        print("  ok    %-14s %s" % (m, getattr(mod, "__version__", "?")))
    except Exception as e:
        print("  GAGAL %-14s %s" % (m, e))


---
## 3. Pengaman konfigurasi

`cek_konfigurasi.py` adalah pengaman struktural yang dipasang setelah
penelitian ini menemukan tujuh kekeliruan yang berjenis sama: dua run dengan
konfigurasi berbeda dibandingkan seolah-olah hanya satu variabel yang berbeda.
Yang paling merusak adalah dua run yang dibandingkan sebagai pasangan
terkontrol padahal satu dijalankan enam epoch dan satunya satu epoch.

Skrip ini mengelompokkan seluruh run lalu berhenti dengan status gagal bila ada
kelompok yang memuat lebih dari satu konfigurasi pelatihan tanpa alasan yang
tercatat. Jalankan lebih dulu; kalau ia gagal, jangan percayai tabel apa pun
sesudahnya.

Keluaran yang diharapkan: 51 kelompok diperiksa, 5 tercampur, kelimanya sudah
terdaftar sebagai pengecualian.


In [ ]:
!python cek_konfigurasi.py


---
# JALUR A, analisis tanpa GPU dan tanpa dataset

Bagian 4 sampai 6. Tidak memerlukan GPU maupun berkas audio.

## 4. Jalankan seluruh skrip pelaporan

Dua puluh sembilan skrip di bawah membangun ulang seluruh tabel hasil, seluruh
grafik, dan kedua PDF, semuanya dari skor yang tersimpan. Diuji memakan 51
detik di mesin lokal, jadi perkirakan satu sampai tiga menit di Colab.

**Empat skrip sengaja tidak dijalankan**, dan alasannya penting:

| Skrip | Alasan |
|---|---|
| `extract_findings.py` | membaca `journal.jsonl` dari sesi workflow lama yang tidak ada di Colab. Skrip inilah yang di mesin lokal menimpa `TEMUAN_RISET.md` sehingga banner penarikan klaim hilang dan 3.609 tanda pisah em masuk kembali. Sengaja dibiarkan tidak berjalan. |
| `hubert_summary.py` | mengandung bug: `glob("runs/hubert*")` ikut menyeret run split acak yang test set-nya 3.574 berkas ke dalam ensemble partisi resmi yang 1.088 berkas |
| `verify_sota_collapse.py` | memerlukan checkpoint Nes2Net-X di `ckpt/` yang tidak ikut repositori |
| `audit.py`, `probe_*.py`, `eval_*.py`, `hard_samples.py`, `analyze_errors.py` | membaca berkas audio, jadi hanya bisa di Jalur B |


In [ ]:
import subprocess, time

SKRIP = [
    "audit_codec.py", "recompute.py", "show.py", "summarize.py", "compare.py",
    "signifikansi.py", "matriks_lr.py", "tabel_ringkas.py", "tabel_2x2.py",
    "dekomposisi.py", "uji_temuan1.py", "uji_kalibrasi.py", "uji_korelasi.py",
    "uji_bandgain_klaim.py", "gen_ablasi.py", "gen_bandgain.py", "gen_report.py",
    "snr_report.py", "crossarch_report.py", "tradeoff_report.py",
    "probe_novelty.py", "best_ensemble.py", "fusion_variants.py",
    "stack_fusion.py", "ab_augfix.py", "make_charts.py", "gambar_paper.py",
    "naskah.py", "make_paper.py",
]

print("%-24s %-7s %6s   %s" % ("skrip", "status", "detik", "baris terakhir"))
print("-" * 92)
gagal = []
for s in SKRIP:
    t0 = time.time()
    # encoding dinyatakan eksplisit: sebagian skrip mencetak karakter di luar
    # ASCII, dan bawaan sistem tidak selalu UTF-8
    p = subprocess.run(["python", s], capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    dt = time.time() - t0
    baris = [b for b in p.stdout.splitlines() if b.strip()]
    akhir = baris[-1][:44] if baris else (p.stderr.strip().splitlines() or [""])[-1][:44]
    st = "ok" if p.returncode == 0 else "GAGAL"
    if p.returncode != 0:
        gagal.append((s, p.stderr.strip().splitlines()[-3:]))
    print("%-24s %-7s %6.1f   %s" % (s, st, dt, akhir))

print("-" * 92)
print(f"{len(SKRIP) - len(gagal)} dari {len(SKRIP)} skrip berhasil")
for s, jejak in gagal:
    print(f"\n--- {s} ---")
    print("\n".join(jejak))


## 5. Apakah hasilnya sama dengan yang tersimpan di repositori?

Ini pemeriksaan yang paling berguna dari Jalur A. `colab/golden_head.json`
memuat sidik jari sha256 tiap berkas markdown pada commit `ef6a2b0`. Sel di
bawah membandingkannya dengan hasil yang baru saja dibangun ulang.

Kalau semuanya identik, berarti setiap angka di repositori benar-benar berasal
dari skor yang ikut disimpan, dan siapa pun dapat membangunnya ulang. Berkas
yang **berubah** menandai tempat berkas hasilnya tertinggal di belakang data
yang sudah ada di `runs/`.

Hasil yang diharapkan pada commit ini: **40 identik, 7 berubah.** Ketujuhnya
memang sudah tidak sinkron di repositori, jadi Colab hanya menampakkannya,
bukan menyebabkannya.


In [ ]:
import hashlib, json, os

g = json.load(open("colab/golden_head.json", encoding="utf-8"))
print("dibandingkan dengan commit", g["komit"], "\n")

sama, beda, hilang = [], [], []
for f, h in sorted(g["berkas"].items()):
    if not os.path.exists(f):
        hilang.append(f)
        continue
    isi = open(f, "rb").read().replace(b"\r\n", b"\n")
    (sama if hashlib.sha256(isi).hexdigest()[:16] == h else beda).append(f)

print(f"identik dengan commit : {len(sama)}")
print(f"berubah setelah dibangun ulang : {len(beda)}")
for f in beda:
    print("    ", f)
if hilang:
    print(f"tidak ada di runtime  : {len(hilang)}")
    for f in hilang:
        print("    ", f)


Kalau daftar "berubah" berisi tujuh berkas berikut, keadaannya persis seperti
di mesin lokal dan tidak ada yang salah dengan Colab:

| berkas | baris yang berbeda |
|---|---|
| `HASIL_ENSEMBLE.md` | 90 |
| `HASIL_NOVELTY_PROBE.md` | 53 |
| `PERBANDINGAN.md` | 48 |
| `HASIL_FUSI.md` | 38 |
| `HASIL_AB_AUGFIX.md` | 14 |
| `HASIL_STACKING.md` | 12 |
| `HASIL_SNR.md` | 2 |

Artinya berkas hasil yang ter-commit **tertinggal di belakang skor yang juga
sudah ter-commit**. Skor delapan seed HuBERT sudah masuk ke `runs/`, tetapi
sebagian dokumen hasil masih memuat angka dari tiga seed. Yang benar adalah
hasil bangunan ulang ini, bukan yang ada di dalam berkas.

Selain itu perhatikan satu sampai tiga tanda pisah em yang muncul kembali di
tiap berkas. `bersihkan_emdash.py` membersihkan **keluaran**, bukan skrip yang
menghasilkannya, sehingga setiap kali skrip dijalankan ulang tanda itu kembali
lagi. Pada `TEMUAN_RISET.md` di mesin lokal jumlahnya mencapai 3.609.

Dua berkas perlu dibaca dengan hati-hati:

- **`HASIL_NOVELTY_PROBE.md`** kehilangan banner "KLAIM INI SUDAH DITARIK" dan
  kembali memuat korelasi yang dihitung atas tiga puluh run tanpa
  merata-ratakan seed per konfigurasi. Perhitungan yang benar ada di
  `HASIL_UJI_KORELASI.md`: sepuluh konfigurasi, r = −0,048, uji permutasi
  p = 0,895. Jangan pakai angka dari `HASIL_NOVELTY_PROBE.md`.
- **`HASIL_AB_AUGFIX.md`** berubah dari n = 3 menjadi n = 14 karena
  `ab_augfix.py` hanya menyaring `b16e10` sehingga menggabungkan tiga strategi
  augmentasi yang berbeda, padahal teksnya tetap menyebut "augmentasi codec,
  seed {42, 1337, 2024}".


## 6. Lihat gambar dan ambil naskahnya

Sembilan gambar di `gambar/` adalah rancangan kedua yang dipakai
`NASKAH.pdf`. Sepuluh gambar di `charts/` adalah rancangan pertama yang masih
dipakai `README.md` dan `PAPER.pdf`.

Perhatikan dua di antaranya: **`charts/01_for_akurasi.png` kosong tanpa satu
batang pun** dan **`charts/06_tradeoff.png` kosong tanpa satu titik pun**,
sedangkan `charts/04_lintas_dataset.png` kehilangan seluruh kolom FoR-2sec.
Sebabnya sama untuk ketiganya: penanda konfigurasi dimasukkan ke kunci
pengelompokan (`"full"` menjadi `"full@b16e10"`), tetapi tempat pencariannya
tidak ikut diperbarui, sehingga `rec.get((m, "official", "full"))` tidak pernah
menemukan apa pun. Seri `gambar/` sudah bebas dari cacat ini.


In [ ]:
import glob
from IPython.display import Image, display, Markdown

display(Markdown("### gambar/  — dipakai NASKAH.pdf"))
for p in sorted(glob.glob("gambar/*.png")):
    display(Markdown("**" + p + "**"))
    display(Image(p, width=780))

display(Markdown("### charts/  — dipakai README.md dan PAPER.pdf"))
KOSONG = {"charts/01_for_akurasi.png": "KOSONG, tidak ada satu batang pun",
          "charts/06_tradeoff.png": "KOSONG, tidak ada satu titik pun",
          "charts/04_lintas_dataset.png": "kolom FoR-2sec hilang"}
for p in sorted(glob.glob("charts/*.png")):
    catatan = KOSONG.get(p.replace("\\", "/"), "")
    display(Markdown(f"**{p}**" + (f"  <- {catatan}" if catatan else "")))
    display(Image(p, width=780))


In [ ]:
# Unduh kedua PDF ke komputer
from google.colab import files
import os

for f in ["NASKAH.pdf", "PAPER.pdf"]:
    if os.path.exists(f):
        print(f, "%.1f MB" % (os.path.getsize(f) / 1e6))
        files.download(f)


---
# JALUR B, melatih model sendiri

Bagian 7 sampai 11. Perlu GPU dan mengunduh dataset 1 GB.

## 7. Dataset: ambil sekali, lalu sama di semua device

Ini bagian yang menentukan apakah angka dari Colab boleh dibandingkan dengan
angka di repositori.

Dataset tidak ikut ke dalam bundle karena ukurannya 1 GB. Ia diunduh dari York
University, dan tidak ada yang menjamin unduhan di device kedua identik dengan
unduhan di device pertama. Unduhan dapat terpotong lalu dilanjutkan dengan cara
yang salah, cermin penyimpanan dapat berbeda, dan penerbitnya dapat memperbarui
arsip tanpa mengubah namanya.

Karena itu dipakai dua pengaman, dan keduanya sudah dihitung dari dataset yang
ada di mesin lokal:

| | nilai |
|---|---|
| sha256 arsip | `acd09881757832b5e9435d93d05e88ed366b41139ba63d47bbfa61344d706a16` |
| ukuran arsip | 1.048.591.372 byte |
| sidik pohon hasil ekstrak | `baf65aaf01f07540145f77d735e0686611f75c1d008f46ad60cff61a15a27a38` |
| jumlah berkas | 17.870 |

Sidik pohon adalah sha256 atas daftar terurut berisi jalur relatif dan sha256
tiap berkas wav. Jalurnya dinormalkan ke bentuk POSIX, sehingga nilainya sama
di Windows maupun di Linux. Nilai lengkapnya ada di `dataset_acuan.json` yang
ikut di dalam bundle, dan sel di bawah **menolak melanjutkan** bila tidak cocok.

Sel di bawah **tidak pernah mengunduh bila arsipnya sudah ada di Drive**, dan
memeriksa dua tempat secara berurutan sebelum menyerah dan mengunduh:

1. `MyDrive/PenelitianAudioDeepfake/Dataset/FoR.zip`, yaitu arsip penelitian
   yang sudah ada di akun Kesari
2. `MyDrive/dataset-for/for-2sec.tar.gz`, yaitu simpanan notebook ini sendiri

Urutan ini penting ketika notebook dijalankan di akun yang sudah memiliki
datasetnya. Tanpa pemeriksaan itu, akan tercipta salinan kedua berukuran 1 GB
di Drive yang sama, dan kuota Drive terpakai dua kali untuk data yang persis
identik.


In [ ]:
import hashlib, json, os, subprocess

ACUAN = json.load(open("dataset_acuan.json", encoding="utf-8"))
SHA_HARAP, BYTES_HARAP = ACUAN["arsip"]["sha256"], ACUAN["arsip"]["bytes"]
URL = "https://bil.eecs.yorku.ca/share/for-2sec.tar.gz"

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"

def sha256(p, blok=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""):
            h.update(b)
    return h.hexdigest()

# Yang sudah ada di Drive dipakai lebih dulu, supaya tidak tercipta salinan
# kedua berukuran 1 GB di akun yang sama.
KANDIDAT = [
    ("arsip penelitian yang sudah ada di Drive ini",
     f"{DRIVE}/PenelitianAudioDeepfake/Dataset/FoR.zip", "zip"),
    ("simpanan notebook ini",
     f"{DRIVE}/dataset-for/for-2sec.tar.gz", "targz"),
]

ARSIP = JENIS = ASAL = None
for nama, path, jenis in KANDIDAT:
    if os.path.exists(path):
        ARSIP, JENIS, ASAL = path, jenis, nama
        print(f"memakai {nama}")
        print(f"  {path}")
        print(f"  {os.path.getsize(path) / 1e9:.2f} GB, tidak ada yang diunduh")
        break

if ARSIP is None:
    print("tidak ada arsip di Drive, mengunduh dari York University ...")
    os.makedirs(f"{DRIVE}/dataset-for", exist_ok=True)
    ARSIP = f"{DRIVE}/dataset-for/for-2sec.tar.gz"
    JENIS, ASAL = "targz", "unduhan baru"
    subprocess.run(["wget", "-c", "--no-verbose", "--show-progress",
                    "-O", ARSIP, URL], check=True)

if JENIS == "targz":
    print("\nmemeriksa sha256 arsip ...")
    besar, sha = os.path.getsize(ARSIP), sha256(ARSIP)
    print(f"  ukuran : {besar}  (acuan {BYTES_HARAP})")
    print(f"  sha256 : {sha}")
    print(f"  acuan  : {SHA_HARAP}")
    assert besar == BYTES_HARAP and sha == SHA_HARAP, (
        "ARSIP BERBEDA dari acuan. Hapus berkas itu lalu ulangi sel ini. Bila "
        "tetap berbeda, penerbitnya mengganti arsip dan angka dari sini tidak "
        "sebanding dengan angka di repositori.")
    print("  ARSIP SAMA dengan yang dipakai di mesin lokal.")
else:
    print("\nPemeriksaan sha256 arsip dilewati. Acuan dibuat dari tar.gz "
          "sedangkan berkas ini zip,")
    print("jadi sidik jari wadahnya memang tidak akan pernah cocok meskipun "
          "isinya sama persis.")
    print("Kesamaan isi diperiksa lewat sidik pohon setelah diekstrak, dan "
          "pemeriksaan itu justru")
    print("lebih dalam karena membandingkan sha256 tiap berkas wav satu per satu.")


In [ ]:
import os, shutil, subprocess, zipfile

TUJUAN = "data/for-2seconds"
TMP = "data/_arsip"
os.makedirs("data", exist_ok=True)
shutil.rmtree(TUJUAN, ignore_errors=True)
shutil.rmtree(TMP, ignore_errors=True)

def cari_akar(mulai):
    # Cari folder yang benar-benar memuat training/real dan testing/fake.
    for dirpath, _, _ in os.walk(mulai):
        if (os.path.isdir(os.path.join(dirpath, "training", "real"))
                and os.path.isdir(os.path.join(dirpath, "testing", "fake"))):
            return dirpath
    return None

if JENIS == "targz":
    # Isi arsip resminya sudah diperiksa: satu folder for-2seconds di tingkat
    # teratas, jadi hasil ekstrak langsung mendarat di tempat yang benar.
    subprocess.run(["tar", "-xzf", ARSIP, "-C", "data"], check=True)
    sumber = cari_akar("data")
else:
    # Zip dapat memiliki lapis folder tambahan, jadi akarnya dicari, bukan
    # ditebak dari nama.
    with zipfile.ZipFile(ARSIP) as z:
        z.extractall(TMP)
    sumber = cari_akar(TMP)

assert sumber, ("struktur training/validation/testing tidak ditemukan. "
                f"isi data/: {os.listdir('data')}")
if os.path.abspath(sumber) != os.path.abspath(TUJUAN):
    shutil.move(sumber, TUJUAN)
shutil.rmtree(TMP, ignore_errors=True)
print(f"dataset siap di {TUJUAN}, sumbernya {ASAL}\n")

total = 0
for s in ["training", "validation", "testing"]:
    for c in ["real", "fake"]:
        n = len(os.listdir(f"{TUJUAN}/{s}/{c}"))
        total += n
        print(f"  {s:<11} {c:<5} {n:>6} berkas")
print(f"  {'TOTAL':<17} {total:>6} berkas   (seharusnya 17.870)")


Terakhir, verifikasi menyeluruh. Sel di bawah menghitung sha256 tiap berkas wav
lalu membandingkan sidik pohonnya dengan acuan. Di mesin lokal ini memakan
sekitar 14 detik untuk 17.870 berkas.

Ini menangkap hal yang tidak tertangkap sidik arsip, misalnya ekstraksi yang
terpotong karena disk penuh, atau folder lain yang tanpa sengaja tercampur ke
dalam `data/for-2seconds`.


In [ ]:
!python cek_dataset.py


Yang diharapkan: lima baris `COCOK` lalu **DATASET SAMA**. Bila salah satu
berbunyi `BEDA`, hentikan di sini. Angka yang dihasilkan dari dataset yang
berbeda tidak boleh dibandingkan dengan angka di repositori, dan itu persis
kelas kekeliruan yang berulang kali terjadi dalam penelitian ini.

Angka partisi yang benar adalah 6.978 / 6.978 untuk training, 1.413 / 1.413
untuk validation, dan 544 / 544 untuk testing, seluruhnya 17.870 berkas.

Perhatikan bahwa partisi resminya **78,1 / 15,8 / 6,1 persen**, bukan
60 / 20 / 20 seperti yang direncanakan proposal. Untuk mendapat 60 / 20 / 20
seluruh data harus digabung lalu dibagi ulang secara acak, dan itulah yang
menghancurkan pemisahan domain yang sengaja dirancang pembuat dataset.


### Apa yang terjadi di tiap keadaan

Bagian 7 dijalankan apa adanya dalam ketiga keadaan di bawah. Tidak ada yang
perlu diubah, dan tidak ada kode yang perlu ditempel.

| Keadaan | Yang dilakukan sel | Waktu | Salinan baru di Drive |
|---|---|---|---|
| Akun sudah punya `PenelitianAudioDeepfake/Dataset/FoR.zip` | memakai zip itu | 2 sampai 3 menit | **tidak ada** |
| Akun sudah punya `dataset-for/for-2sec.tar.gz` | memakai tar.gz itu | 1 sampai 2 menit | tidak ada |
| Akun belum punya apa-apa | mengunduh dari York, lalu menyimpan ke `dataset-for/` | 5 sampai 10 menit | satu, 1 GB |

Baris pertama itu yang berlaku ketika notebook dijalankan di akun yang sudah
memiliki datasetnya. Tanpa pemeriksaan berurutan tadi, akan tercipta salinan
kedua berukuran 1 GB untuk data yang persis identik.

### Cara memastikannya ketika sumbernya berbeda

Sidik jari arsip hanya berlaku untuk tar.gz resmi. Zip memiliki wadah yang
berbeda sehingga sha256-nya tidak akan pernah cocok meskipun isinya sama
persis, dan sel akan mengatakannya terus terang alih-alih berpura-pura lolos.

Yang menjawab pertanyaan sebenarnya adalah **sidik pohon** pada bagian
berikutnya, karena ia dihitung dari berkas wav hasil ekstrak, bukan dari
wadahnya. Kalau sidik itu cocok, kedua salinan terbukti identik berkas demi
berkas, dari mana pun asalnya.

Ini bukan sekadar kenyamanan. Bila dua orang mengerjakan penelitian yang sama
di komputer yang berbeda, sidik pohon yang cocok adalah bukti bahwa angka
keduanya memang sebanding. Tanpa itu, ketika hasilnya berbeda, tidak ada yang
tahu apakah penyebabnya metodenya atau datanya.


## 8. Bangun ulang manifest, lalu pasang patch Colab

**Membangun ulang manifest wajib.** `manifest.csv` yang ikut di dalam bundle
dibuat di Windows dan memuat pemisah jalur backslash
(`data/for-2seconds\training\real\file1000...`). Di Linux itu terbaca sebagai
satu nama berkas panjang, sehingga `soundfile` akan gagal membuka setiap
berkas. Untuk Jalur A hal itu tidak masalah karena `naskah.py` dan
`gambar_paper.py` hanya memakai kolom split, label, dan is_mp3, bukan kolom
path.

Sesudahnya, patch Colab mengganti pemilihan presisi di `train.py`. Baris
aslinya memaksa bfloat16 di setiap GPU. Itu benar pada RTX 5060 Ti yang
berarsitektur Blackwell, tetapi Tesla T4 tidak punya bfloat16 di perangkat
kerasnya, dan PyTorch akan meng-emulasinya tanpa satu pun peringatan. Patch
ini membaca compute capability kartu lalu memilih float16 beserta GradScaler
bila perlu. Pada kartu yang memang mendukung bfloat16, perhitungannya tetap
persis sama seperti semula.

Patch menyimpan cadangan dan dapat dibatalkan kapan saja dengan
`python colab/colab_patch.py --revert`.


In [ ]:
import os, sys

if os.path.exists("manifest.csv"):
    os.remove("manifest.csv")
sys.path.insert(0, ".")
from forlib.data import build_manifest

baris = build_manifest("data/for-2seconds", "manifest.csv")
print(len(baris), "baris manifest dibangun ulang")
print("contoh path:", baris[0]["path"])

mp3 = sum(r["is_mp3"] for r in baris if r["split_official"] == "training" and r["label"] == 1)
tot = sum(1 for r in baris if r["split_official"] == "training" and r["label"] == 1)
mp3u = sum(r["is_mp3"] for r in baris if r["split_official"] == "testing" and r["label"] == 1)
totu = sum(1 for r in baris if r["split_official"] == "testing" and r["label"] == 1)
print(f"\nkebocoran codec, dihitung ulang dari berkas Anda sendiri:")
print(f"  palsu di data latih yang berasal MP3 : {mp3}/{tot} = {100*mp3/tot:.1f} persen")
print(f"  palsu di data uji yang berasal MP3   : {mp3u}/{totu} = {100*mp3u/totu:.1f} persen")


In [ ]:
!python colab/colab_patch.py --apply
!python colab/colab_patch.py --status


## 9. Latihan pertama, cepat

`cnn_asp` adalah CNN dengan attentive statistics pooling, 1,54 juta parameter,
tanpa encoder pra-latih sehingga tidak perlu mengunduh apa pun dari
HuggingFace. Ini pilihan yang tepat untuk mencoba karena selesai paling cepat.

Tiga hal pada perintah di bawah yang sengaja berbeda dari perintah di README:

- `--out runs_colab` menaruh hasilnya di direktori terpisah. **Jangan pernah
  menulis ke `runs/`.** Run Colab memakai presisi dan perangkat keras yang
  berbeda, dan menggabungkannya dengan run lokal akan melaporkan ragam antar
  perangkat sebagai ragam antar inisialisasi acak. Itu persis kelas kekeliruan
  yang tujuh kali terjadi dalam penelitian ini.
- `--workers 2` karena Colab tingkat gratis hanya punya dua core, sedangkan
  bawaannya enam.
- `--seed 42` supaya dapat dibandingkan dengan run lokal yang sudah ada.

Perkiraan waktu di T4: sekitar 1,5 sampai 2 menit per epoch, jadi 15 sampai 20
menit untuk sepuluh epoch.


In [ ]:
!python train.py --model cnn_asp --split official --augment codec \
                 --epochs 10 --batch 32 --workers 2 --seed 42 \
                 --out runs_colab


## 10. Bandingkan dengan run lokal yang setara

Run lokal pembandingnya adalah `runs/cnn_asp_official_codec_s42`, yang memakai
konfigurasi identik: partisi resmi, augmentasi codec, sepuluh epoch, batch 32,
seed 42. Yang berbeda hanya perangkat keras dan presisinya.

Angkanya tidak akan sama persis, dan itu memang yang diharapkan. Yang perlu
diperhatikan bukan selisihnya, melainkan apakah selisih itu lebih besar
daripada ragam antar inisialisasi acak pada konfigurasi ini, yaitu **±3,50
poin persentase** atas tiga seed. Kalau selisihnya di bawah itu, tidak ada
yang bisa disimpulkan darinya.


In [ ]:
import glob, sys
import numpy as np

sys.path.insert(0, ".")
from forlib.metrics import full_metrics, prior_matched_threshold

def ringkas(d):
    y, p, _ = np.load(f"{d}/test_scores.npy")
    y = y.astype(int)
    return (full_metrics(y, p, 0.5), full_metrics(y, p, prior_matched_threshold(p, 0.5)))

print(f"{'run':<42}{'acc@0,5':>9}{'acc@prior':>11}{'AUC':>9}{'EER':>9}   asal")
print("-" * 84)
for d in sorted(glob.glob("runs_colab/*")) + ["runs/cnn_asp_official_codec_s42"]:
    try:
        m0, mp = ringkas(d)
    except Exception as e:
        print(f"{d:<42}  {e}")
        continue
    asal = "Colab" if d.startswith("runs_colab") else "lokal"
    nama = d.split("/")[-1]
    print(f"{nama:<42}{m0['accuracy']*100:>8.2f}%{mp['accuracy']*100:>10.2f}%"
          f"{m0['auc']:>9.4f}{m0['eer']*100:>8.2f}%   {asal}")
print("-" * 84)
print("ragam antar inisialisasi pada konfigurasi ini: +/-3,50 pp atas 3 seed")
print("selisih yang lebih kecil dari itu tidak dapat ditafsirkan")


## 11. Latihan kedua, opsional dan jauh lebih lama

WavLM Large adalah model 300 juta parameter dengan encoder yang dibekukan,
sehingga yang dilatih hanya sekitar 1,2 juta parameter di bagian belakangnya.
Konfigurasi ini yang mencapai angka tertinggi pada partisi resmi dalam
penelitian ini, yaitu **98,36 persen ± 0,63** atas lima inisialisasi.

Yang perlu disiapkan: unduhan checkpoint sekitar 1,2 GB dari HuggingFace pada
epoch pertama, batch diturunkan ke 16 supaya muat di 16 GB VRAM, dan waktu
sekitar 8 sampai 12 menit per epoch di T4. Sepuluh epoch berarti satu setengah
sampai dua jam, jadi pastikan tab Colab tetap terbuka.

Kalau hanya ingin memastikan pipeline-nya jalan, turunkan `--epochs` menjadi 1.


In [ ]:
# Perkiraan 1,5 sampai 2 jam di T4. Turunkan --epochs jadi 1 untuk uji cepat.
!python train.py --model wavlm --split official --augment full \
                 --epochs 10 --batch 16 --workers 2 --seed 42 \
                 --out runs_colab


Beberapa varian lain yang menarik dicoba, semuanya menulis ke `runs_colab`:

```bash
# menunjukkan kebocoran domain: split acak memberi akurasi hampir 100 persen
python train.py --model cnn_asp --split random --augment none \
                --epochs 10 --batch 32 --workers 2 --seed 42 --out runs_colab

# tanpa augmentasi pada partisi resmi: model menandai semuanya sebagai asli
python train.py --model cnn_asp --split official --augment none \
                --epochs 10 --batch 32 --workers 2 --seed 42 --out runs_colab

# mereproduksi temuan learning rate: laju 0,001 seragam menjatuhkan WavLM
python train.py --model wavlm --split official --augment proposal \
                --uniform-lr 0.001 --normalize peak --epochs 20 --batch 16 \
                --workers 2 --seed 42 --out runs_colab

# usulan band-gain, ablasi variabel tunggal di atas augmentasi penuh
python train.py --model nes2net --split official --augment fullbg \
                --epochs 10 --batch 16 --workers 2 --seed 42 --out runs_colab
```

Dua perintah pertama bersama-sama adalah temuan pembuka penelitian ini. Tetapi
bacalah `HASIL_TEMUAN1.md` sebelum menafsirkan selisihnya: dari 49,94 poin yang
sempat dilaporkan, hanya **6,92 poin** yang benar-benar berasal dari protokol
pembagian data, dan itu pun belum terbukti (p = 0,0822). Sebanyak 42,52 poin
berasal dari ambang keputusan yang tidak lagi cocok.


---
## 12. Simpan hasil sebelum sesi berakhir

Runtime Colab dihapus setelah terputus, jadi apa pun yang tidak disalin keluar
akan hilang. Sesi gratis berhenti setelah kira-kira 90 menit menganggur atau
12 jam berjalan.

Bobot model (`best.pt`) sengaja tidak ikut disalin. Ukurannya bisa 1,2 GB per
run untuk model besar, sedangkan seluruh analisis hanya memerlukan
`test_scores.npy` dan `results.json` yang berukuran puluhan kilobyte.


In [ ]:
import glob, os, shutil

def kumpulkan(tujuan):
    # Salin hasil ke `tujuan`, tanpa bobot model yang berukuran gigabyte.
    os.makedirs(tujuan, exist_ok=True)
    n = 0
    for pola in ["runs_colab/*/results.json", "runs_colab/*/test_scores.npy"]:
        for src in glob.glob(pola):
            dst = os.path.join(tujuan, src)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    for pola in ["*.md", "*.pdf", "gambar/*.png", "charts/*.png"]:
        for src in glob.glob(pola):
            dst = os.path.join(tujuan, src)
            os.makedirs(os.path.dirname(dst) or tujuan, exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    besar = sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(tujuan) for f in fs)
    print(f"{n} berkas disalin ke {tujuan}  ({besar/1e6:.1f} MB)")

# Cara 1, ke Google Drive
from google.colab import drive
drive.mount("/content/drive")
kumpulkan("/content/drive/MyDrive/general-ai-hasil-colab")


In [ ]:
# Cara 2, unduh satu berkas zip ke komputer
import os, shutil
from google.colab import files

shutil.rmtree("/content/keluar", ignore_errors=True)
kumpulkan("/content/keluar")
shutil.make_archive("/content/hasil-colab", "zip", "/content/keluar")
print("%.1f MB" % (os.path.getsize("/content/hasil-colab.zip") / 1e6))
files.download("/content/hasil-colab.zip")


---
## 13. Yang perlu diingat sebelum memakai angka dari sini

1. **Jangan pernah menyalin `runs_colab/` ke dalam `runs/`.** Presisi dan
   perangkat kerasnya berbeda. Menggabungkannya akan melaporkan ragam antar
   perangkat sebagai ragam antar inisialisasi acak, dan `cek_konfigurasi.py`
   tidak dapat menangkapnya karena ia hanya memeriksa epoch dan batch.

2. **Satu run tidak dapat dijadikan peringkat.** Ragam antar inisialisasi pada
   32 konfigurasi yang dijalankan minimal tiga kali memiliki median 1,80 poin
   persentase, dan pada AST mencapai 7,0 poin. Sebagian besar selisih antar
   konfigurasi yang menarik berukuran lebih kecil daripada itu. Minimal tiga
   seed, lebih baik lima.

3. **Ambang prior-matched bersifat transduktif.** Ia memerlukan seluruh skor
   uji sekaligus beserta proporsi kelas yang sebenarnya. Sah untuk forensik
   arsip, tidak sah untuk deteksi satu per satu. Kolom `acc@0,5` adalah angka
   induktifnya, dan selisih keduanya justru merupakan salah satu temuan utama
   penelitian ini.

4. **Dua temuan yang bertahan tidak menyangkut arsitektur sama sekali**, yaitu
   kalibrasi ambang keputusan dan besaran learning rate relatif terhadap
   encoder. Tabel status tiap temuan ada di `README.md`, dan daftar tiga belas
   klaim yang ditarik ada di bagian 7 `NASKAH.pdf`.

5. **Baca `NASKAH.pdf` lebih dulu.** Seluruh angka di dalamnya dihitung ulang
   dari berkas hasil setiap kali dokumen dibangun, sehingga tidak mungkin ada
   angka usang. Berkas `.md` yang lain adalah catatan tahapan, dan enam di
   antaranya membawa banner penarikan karena memuat angka yang sudah tidak
   berlaku.
